# Brain MRI Segmentation Experimentation
This notebook demonstrates the usage of the modular U-Net implementation for brain MRI segmentation.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from configs.config import get_config
from datasets.brain_dataset import BrainMRIDataset
from models.factory import ModelFactory
from losses.dice_bce_loss import DiceBCELoss
from trainers.trainer import Trainer
from visualization.visualizer import visualize_predictions
from utils.generate_synthetic_data import create_synthetic_data

config = get_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Dataset Preparation
First, we generate synthetic data for demonstration purposes.

In [ ]:
create_synthetic_data(root_dir=config.data_root)

dataset = BrainMRIDataset(
    root_dir=config.data_root,
    modalities=config.modalities,
    image_size=config.image_size,
    remove_empty_slices=config.remove_empty_slices
)

train_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)
print(f"Dataset loaded with {len(dataset)} slices.")

## 2. Model Selection
We can easily swap between `unet` and `attention_unet` using the `ModelFactory`.

In [ ]:
# Swap "unet" for "attention_unet" here
model_name = "attention_unet"
model = ModelFactory.create(
    model_name=model_name,
    in_channels=config.in_channels,
    out_channels=config.out_channels
).to(device)
print(f"Created model: {model_name}")

## 3. Training
We use the generic trainer to train our model.

In [ ]:
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=train_loader, # Using train as val for demo
    criterion=criterion,
    optimizer=optimizer,
    config=config,
    device=device
)

history = trainer.fit()

## 4. Visualization
Finally, we visualize a prediction.

In [ ]:
model.eval()
with torch.no_grad():
    image, mask = dataset[0]
    image = image.unsqueeze(0).to(device)
    pred = model(image)
    
    visualize_predictions(image[0], mask, pred[0])